# OC14 — Eval of the SFT (Base) model on the 300-case LLM-consensus gold

Loads the LoRA adapter from the SFT kernel's output and generates with the **correct**
inference config: stop on `<|im_end|>` (the small model otherwise runs past the answer into
repetition), and the **exact trained system prompt** (read back from the dataset). Scores the
**300-case stratified eval-gold** (`triage_eval_gold.jsonl`, 100/100/100, 3-model consensus).
Headline = **macro-F1** (robust to the maximale-skewed class prior) + **maximale recall** (safety)
+ a confusion matrix.

In [ ]:
# Official Unsloth Kaggle install (verbatim, June 2026): install torch from the cu128
# index FIRST so its wheels carry the T4/sm_75 CUDA kernels. Plain `pip install unsloth`
# let pip resolve a torch build lacking them -> 'CUDA: no kernel image' at model load.
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import subprocess, sys
with open('/kaggle/working/requirements-train.lock.txt', 'w') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh)
print('lockfile written ->', '/kaggle/working/requirements-train.lock.txt')

In [ ]:
import glob, os, json
# Prefer the SFT+DPO adapter if attached; else the SFT adapter. (Compare both runs.)
ad = (glob.glob('/kaggle/input/**/dpo_adapter/adapter_config.json', recursive=True)
      or glob.glob('/kaggle/input/**/sft_adapter/adapter_config.json', recursive=True))
assert ad, 'adapter not found — attach the SFT and/or DPO kernel as a kernel source'
ADAPTER_DIR = os.path.dirname(ad[0]); print('ADAPTER_DIR =', ADAPTER_DIR)
dd = glob.glob('/kaggle/input/**/sft_train.jsonl', recursive=True)
DATA_DIR = os.path.dirname(dd[0]) if dd else None
# Read the EXACT trained system prompts back from the data (guarantees eval matches training).
SYS = {}
if DATA_DIR:
    for ln in open(f'{DATA_DIR}/sft_train.jsonl', encoding='utf-8').read().split('\n'):
        if not ln.strip():
            continue
        r = json.loads(ln)
        SYS.setdefault(r['lang'], r['messages'][0]['content'])
        if len(SYS) == 2:
            break
print('system prompts loaded for langs:', list(SYS))

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
IM_END = tokenizer.convert_tokens_to_ids('<|im_end|>')
STOP_IDS = [IM_END, tokenizer.eos_token_id]
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print('eos:', tokenizer.eos_token, '| im_end id:', IM_END)
def gen_batch(systems, users):
    # greedy (deterministic, reproducible) + left-padded batch — standard for classification-style eval
    texts = [tokenizer.apply_chat_template(
                 [{'role': 'system', 'content': s}, {'role': 'user', 'content': u}],
                 add_generation_prompt=True, enable_thinking=False, tokenize=False)
             for s, u in zip(systems, users)]
    enc = tokenizer(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=128, do_sample=False, repetition_penalty=1.1,
                         eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id)
    plen = enc['input_ids'].shape[1]
    return [tokenizer.decode(o[plen:], skip_special_tokens=True).strip() for o in out]

In [ ]:
LEVELS = ('urgence maximale', 'urgence modérée', 'urgence différée')
import re as _re
_VERDICT = _re.compile(r'(?:niveau d.urgence|urgency level)\s*:?\s*(.{0,40})', _re.IGNORECASE)
def extract_urgency(t):
    low = (t or '').lower(); m = _VERDICT.search(low)
    if m:
        seg = m.group(1); hits = [(seg.index(l), l) for l in LEVELS if l in seg]
        if hits:
            return min(hits)[1]
    hits = [(low.index(l), l) for l in LEVELS if l in low]
    return min(hits)[1] if hits else None
def has_disclaimer(t):
    low = t.lower(); return ('ne remplace pas' in low) or ('does not replace' in low)
# --- inlined triage_report (mirror of src/oc14_triage/eval/metrics.py) ---
def triage_report(pairs):
    from collections import Counter
    pairs = [(p, g) for p, g in pairs if g in LEVELS]; n = len(pairs)
    if not n:
        return {'n': 0}
    def _wilson(k, m, z=1.96):
        if not m:
            return None
        p = k/m; d = 1 + z*z/m; c = p + z*z/(2*m)
        h = z*((p*(1-p)/m + z*z/(4*m*m))**0.5)
        return (round((c-h)/d, 3), round((c+h)/d, 3))
    conf = Counter((g, p) for p, g in pairs); rec, prec, f1, rci = {}, {}, {}, {}
    for lv in LEVELS:
        ng = sum(g == lv for p, g in pairs); npr = sum(p == lv for p, g in pairs)
        tp = sum(p == lv for p, g in pairs if g == lv)
        rec[lv] = round(tp/ng, 3) if ng else None; rci[lv] = _wilson(tp, ng)
        prec[lv] = round(tp/npr, 3) if npr else None
        r_, p_ = rec[lv], prec[lv]
        f1[lv] = round(2*p_*r_/(p_+r_), 3) if (p_ and r_) else (0.0 if (p_ == 0 or r_ == 0) else None)
    present = [lv for lv in LEVELS if any(g == lv for p, g in pairs)]
    macro = lambda d: round(sum(d[lv] or 0 for lv in present)/len(present), 3) if present else None
    out = {'n': n, 'accuracy': round(sum(p == g for p, g in pairs)/n, 3),
           'recall_per_level': rec, 'precision_per_level': prec, 'f1_per_level': f1,
           'macro_recall': macro(rec), 'macro_precision': macro(prec), 'macro_f1': macro(f1),
           'recall_urgence_maximale': rec['urgence maximale'], 'recall_ci_per_level': rci,
           'confusion_gold_pred': {f'{g}->{p}': c for (g, p), c in sorted(conf.items())}}
    try:
        from sklearn.metrics import cohen_kappa_score
        out['cohen_kappa'] = round(cohen_kappa_score([g for p, g in pairs], [p for p, g in pairs], labels=list(LEVELS)), 3)
    except Exception:
        out['cohen_kappa'] = None
    return out
gp = glob.glob('/kaggle/input/**/triage_eval_gold.jsonl', recursive=True)
assert gp, 'triage_eval_gold.jsonl not found — version the dataset with the gold file'
gold = [json.loads(l) for l in open(gp[0], encoding='utf-8').read().split('\n') if l.strip()]
print('eval-gold:', len(gold), 'cases')
SYSFR = SYS.get('fr') or 'Tu es un assistant de triage médical.'
pairs, beh = [], []
B = 16
for i in range(0, len(gold), B):
    chunk = gold[i:i+B]
    outs = gen_batch([SYSFR]*len(chunk), [r['user'] for r in chunk])
    for r, txt in zip(chunk, outs):
        pred = extract_urgency(txt)
        pairs.append((pred, r['gold_urgency']))
        beh.append((has_disclaimer(txt), pred is not None, '<think>' not in txt.lower()))
    print(f'  {min(i+B, len(gold))}/{len(gold)} generated')
rep = triage_report(pairs)
print('\n=== TRIAGE REPORT on the 300-case gold (macro_f1 = headline) ===')
print(json.dumps(rep, ensure_ascii=False, indent=2))
nb = len(beh) or 1
print('behavioural: disclaimer=%.2f format=%.2f no_think=%.2f' % (
    sum(b[0] for b in beh)/nb, sum(b[1] for b in beh)/nb, sum(b[2] for b in beh)/nb))